In [ ]:
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader

# Paths
train_dir = "/kaggle/input/q1-stage-3-2026/PlantVillage/train"
test_dir  = "/kaggle/input/q1-stage-3-2026/PlantVillage/test"

# Transforms
train_tf = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_tf = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Datasets
train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
test_ds  = datasets.ImageFolder(test_dir,  transform=test_tf)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

print("Classes:", train_ds.classes)
print("Train samples:", len(train_ds))
print("Test samples:", len(test_ds))


In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

images, labels = next(iter(train_loader))

plt.figure(figsize=(8,4))
for i in range(6):
    plt.subplot(2,3,i+1)
    img = (images[i] * std + mean).permute(1,2,0).clamp(0,1)
    plt.imshow(img)
    plt.title(train_ds.classes[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Write your code here
import torch
import torch.nn as nn

class CNN5(nn.Module):
    def __init__(self, num_classes):
        super(CNN5, self).__init__()

        # 5 Convolutional Layers
        self.conv1 = nn.Conv2d(in_channels=3,  out_channels=16,  kernel_size=3, stride=1, padding=1)
        self.bn1   = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32,  kernel_size=3, stride=1, padding=1)
        self.bn2   = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64,  kernel_size=3, stride=1, padding=1)
        self.bn3   = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn4   = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(in_channels=128,out_channels=256, kernel_size=3, stride=1, padding=1)
        self.bn5   = nn.BatchNorm2d(256)

        # Activation + Pooling
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # For input 32x32:
        # After 5 pools: 32->16->8->4->2->1
        self.fc1 = nn.Linear(256 * 1 * 1, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))  # (B,16,16,16)
        x = self.pool(self.relu(self.bn2(self.conv2(x))))  # (B,32,8,8)
        x = self.pool(self.relu(self.bn3(self.conv3(x))))  # (B,64,4,4)
        x = self.pool(self.relu(self.bn4(self.conv4(x))))  # (B,128,2,2)
        x = self.pool(self.relu(self.bn5(self.conv5(x))))  # (B,256,1,1)

        x = torch.flatten(x, start_dim=1)  # (B,256)

        x = self.relu(self.fc1(x))         # (B,128)
        x = self.fc2(x)                    # (B,num_classes)
        return x

# Example
model = CNN5(num_classes=len(train_ds.classes))
print(model)


In [ ]:
# Write your code here
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Part 3: Training and Validation Functions
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)                 # logits
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
# Part 4: Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN5(num_classes=len(train_ds.classes)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5

train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")


In [ ]:
# Plot Loss
plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

# Plot Accuracy
plt.figure()
plt.plot(train_accs, label="Train Acc")
plt.plot(val_accs, label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

In [ ]:
# Write your code here
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

class CNN5_Residual(nn.Module):
    def __init__(self, num_classes):
        super(CNN5_Residual, self).__init__()

        # Conv layers + BN
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1);   self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, 1, 1);  self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3, 1, 1);  self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 128, 3, 1, 1); self.bn4 = nn.BatchNorm2d(128)
        self.conv5 = nn.Conv2d(128, 256, 3, 1, 1);self.bn5 = nn.BatchNorm2d(256)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)

        # Skip:
        self.skip_proj = nn.Conv2d(32, 128, kernel_size=1)
        self.skip_pool = nn.MaxPool2d(kernel_size=4, stride=4)  # 8x8 -> 2x2


        self.fc1 = nn.Linear(256 * 1 * 1, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Layer 1
        x = self.pool(self.relu(self.bn1(self.conv1(x))))   # (B,16,16,16)

        # Layer 2
        x2 = self.pool(self.relu(self.bn2(self.conv2(x))))     # (B,32,8,8)

        # Layer 3
        x3 = self.pool(self.relu(self.bn3(self.conv3(x2))))   # (B,64,4,4)

        # Layer 4
        x4 = self.pool(self.relu(self.bn4(self.conv4(x3))))     # (B,128,2,2)

        # Residual from layer 2 -> layer 4 (summation)
        skip = self.skip_pool(self.skip_proj(x2))            # (B,128,2,2)
        x4 = x4 + skip

        # Layer 5
        x5 = self.pool(self.relu(self.bn5(self.conv5(x4)))) # (B,256,1,1)

        x5 = torch.flatten(x5, start_dim=1)                 # (B,256)
        x5 = self.relu(self.fc1(x5))
        x5 = self.fc2(x5)
        return x5


# Retrain model with residual connection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_res = CNN5_Residual(num_classes=len(train_ds.classes)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_res.parameters(), lr=1e-3)

epochs = 5
train_losses_res, val_losses_res = [], []
train_accs_res, val_accs_res = [], []

for epoch in range(epochs):
    tl, ta = train_one_epoch(model_res, train_loader, optimizer, criterion, device)
    vl, va = validate(model_res, test_loader, criterion, device)

    train_losses_res.append(tl)
    val_losses_res.append(vl)
    train_accs_res.append(ta)
    val_accs_res.append(va)

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {tl:.4f} | Train Acc: {ta:.4f} "
          f"Val Loss: {vl:.4f} | Val Acc: {va:.4f}")


In [ ]:
# Plot Loss
plt.figure()
plt.plot(train_losses_res, label="Train Loss")
plt.plot(val_losses_res, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Residual Model: Training vs Validation Loss")
plt.legend()
plt.show()

# Plot Accuracy
plt.figure()
plt.plot(train_accs_res, label="Train Acc")
plt.plot(val_accs_res, label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Residual Model: Training vs Validation Accuracy")
plt.legend()
plt.show()
